In [1]:
import os
import pandas as pd
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

c:\Users\rites\OneDrive\Desktop\JobDescriptionRAG\rag_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rites\AppData\Local\Temp\ipykernel_21048\37926496.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
dataset = load_dataset("batuhanmtl/job-skill-set")

df = dataset["train"].to_pandas()

print(df.shape)
df.sample(5)

(1167, 5)


,job_id,category,job_title,job_description,job_skill_set
1000,3902326947,BUSINESS-DEVELOPMENT,Business Development Representative,POSITION PURPOSE\nTHE BUSINESS DEVELOPMENT REP...,"['sales skills', 'client presentations', 'nego..."
1118,3904704593,BUSINESS-DEVELOPMENT,Enterprise Business Development Representative,"THE ROLE\n\nHERE AT MARIGOLD, WE’RE WORKING TO...","['Communication', 'Self-Motivation', 'Collabor..."
1052,3884823083,BUSINESS-DEVELOPMENT,Area Director Business Development,SUNRISE SENIOR LIVING WAS AGAIN CERTIFIED AS A...,"['business development', 'account management',..."
823,3894894372,FINANCE,Finance Manager,THE IDEAL CANDIDATE WILL BE RESPONSIBLE FOR WO...,"['financial strategies', 'data analysis', 'fin..."
895,3887479396,FINANCE,Finance Business Partner Specialty Care for In...,COMPANY DESCRIPTION\n\nABBVIE'S MISSION IS TO ...,"['financial analysis', 'budgeting', 'accountin..."


In [18]:
print(df.columns)

df.sample(8)

Index(['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'], dtype='str')


,job_id,category,job_title,job_description,job_skill_set
142,3903487939,HR,Human Resources Business Partner,JOB TITLE: HR BUSINESS PARTNERLOCATION: HERNDO...,"['human resources expertise', 'employee relati..."
1122,3903476020,BUSINESS-DEVELOPMENT,Engineering Business Development Specialist In...,OVERVIEW\n\nWE ARE SEEKING A FULL-TIME ENGINEE...,"['Microsoft Word', 'Microsoft Excel', 'Microso..."
550,3904988667,SALES,Salesperson,JOB DESCRIPTION\n\nWHAT IS A SALESPERSON?\n\nE...,"['inventory management', 'communication', 'cus..."
685,3904099287,SALES,Outside Sales Representative,INDUSTRY | INDUSTRIAL - SERVICESLOCATION | HAG...,"['communication', 'relationship building', 'ti..."
783,3903843298,FINANCE,Senior Finance Manager,THE AEROSPACE CORPORATION IS THE TRUSTED PARTN...,"['data analytics', 'team management', 'leaders..."
632,3904963149,SALES,Sales Associate / Jewelry Consultant - Kay Jew...,WE HAVE MANY OPPORTUNITIES AVAILABLE ON OUR OT...,"['customer service', 'sales', 'retail experien..."
431,3901345936,INFORMATION-TECHNOLOGY,Information Technology Security Analyst,IT SECURITY 2 (GENERAL ANALYST)THE IT SECURITY...,"['network engineering', 'technical reporting',..."
1072,3885866733,BUSINESS-DEVELOPMENT,Business Development Representative,INDUSTRY | SERVICES - BUSINESS SERVICESLOCATIO...,"['B2B Sales', 'Consultative Selling', 'Account..."


In [4]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():

    doc = Document(
        page_content=(
            f"Job Title: {row['job_title']}\n"
            f"Category: {row['category']}\n"
            f"Skills: {row['job_skill_set']}\n"
            f"Job Description: {row['job_description']}"
        ),
        metadata={
            "job_id": str(row["job_id"]),
            "category": str(row["category"]),
            "job_title": str(row["job_title"]),
            "job_skill_set": str(row["job_skill_set"])
        }
    )

    documents.append(doc)

print("Total documents:", len(documents))
print(documents[0])

Total documents: 1167
page_content='Job Title: Sr Human Resource Generalist
Category: HR
Skills: ['employee relations', 'talent acquisition', 'performance management', 'compensation and benefits', 'employment laws and regulations', 'analytical skills', 'employee benefits administration', 'HR policies and procedures', 'recruitment strategies', 'FMLA', 'ADA', 'communication', 'interpersonal skills', 'collaboration', 'influence', 'change management', 'professional development', 'payroll processing', 'job design', 'salary administration', 'human capital management system', 'HR audits and assessments', 'FLSA', 'ERISA', 'EEOC guidelines', 'confidentiality', 'intellectual curiosity']
Job Description: SUMMARY
THE SR. HR GENERALIST PROVIDES HR EXPERTISE ACROSS VARIOUS FUNCTIONS INCLUDING EMPLOYEE RELATIONS, TALENT ACQUISITION, PERFORMANCE MANAGEMENT, PAYROLL, COMPENSATION AND BENEFITS. THE SR. HR GENERALIST HAS RESPONSIBILITIES THAT ARE STRATEGIC, ADMINISTRATIVE, AND CONSULTATIVE. THIS ROLE PRO

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 14860
page_content='Job Title: Sr Human Resource Generalist
Category: HR' metadata={'job_id': '3902668440', 'category': 'HR', 'job_title': 'Sr Human Resource Generalist', 'job_skill_set': "['employee relations', 'talent acquisition', 'performance management', 'compensation and benefits', 'employment laws and regulations', 'analytical skills', 'employee benefits administration', 'HR policies and procedures', 'recruitment strategies', 'FMLA', 'ADA', 'communication', 'interpersonal skills', 'collaboration', 'influence', 'change management', 'professional development', 'payroll processing', 'job design', 'salary administration', 'human capital management system', 'HR audits and assessments', 'FLSA', 'ERISA', 'EEOC guidelines', 'confidentiality', 'intellectual curiosity']"}


In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2215.56it/s]


In [7]:
vector_store = FAISS.from_documents( chunks, embeddings )

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [8]:
vector_store.save_local("faiss_index")
print("FAISS index saved successfully!")

FAISS index saved successfully!


In [9]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k": 4})
query = "What skills are required for a Data Scientist?"

results = retriever.invoke(query)

for doc in results:
    print("Job Title:", doc.metadata.get("job_title"))
    print("Category:", doc.metadata.get("category"))
    print("Skills:", doc.metadata.get("job_skill_set"))
    print("Description:", doc.page_content[:500])
    print("-" * 50)


Job Title: Information Technology Specialist - Data Operations
Category: INFORMATION-TECHNOLOGY
Skills: ['data management', 'database management systems', 'data security', 'system architecture', 'integration techniques', 'testing methods', 'communication', 'problem solving', 'analytical thinking', 'adaptability', 'project management', 'team collaboration', 'technical documentation', 'data analysis', 'cloud reporting and analytics', 'data quality standards', 'data mining', 'data warehousing', 'information assurance certification (e.g., Sec+)']
Description: Category: INFORMATION-TECHNOLOGY
Skills: ['data management', 'database management systems', 'data security', 'system architecture', 'integration techniques', 'testing methods', 'communication', 'problem solving', 'analytical thinking', 'adaptability', 'project management', 'team collaboration', 'technical documentation', 'data analysis', 'cloud reporting and analytics', 'data quality standards', 'data mining', 'data warehousing', 'inf

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

In [11]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [31]:
chat_template = ChatPromptTemplate([
    (
        "system",
        "You are a helpful job information expert. "
       
    ),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    )
])

In [32]:
from dotenv import load_dotenv
load_dotenv()
model = ChatGoogleGenerativeAI( model="gemini-3.5-flash-lite")

In [33]:
from langchain_core.output_parsers import StrOutputParser
rag_chain = (
    {
        "context": retriever | format_docs, "question": RunnablePassthrough()
    }| chat_template| model | StrOutputParser()
)

In [34]:
response = rag_chain.invoke("What skills are required for a Data Scientist?")

print(response)

Based on the provided context, the required and related skills for data-related roles (such as Data Scientists, Data Analysts, and Database/Data Management professionals) include:

**Technical Skills:**
* **Data & Database Management:** Data management, database management systems, data integration, metadata management, normalization, data warehousing, data mining, and storage capacity planning.
* **Data Analysis & Processing:** Data analysis (including strong Excel skills), ability to analyze and manipulate large quantities of data, and data quality standards.
* **Cloud & Reporting:** Cloud reporting and analytics.
* **Architecture & Security:** System/systems architecture, data security, integration techniques, testing methods, backup and recovery, and information assurance certification (e.g., Sec+ / SEC+).
* **Technical Operations:** Query languages, technical documentation, feasibility studies, and cost analysis.
* **Software & Tools:** Proficient with Microsoft tools (Word, Power